# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **hoàn chỉnh** được thiết kế chạy trên **Google Colab (hoặc Kaggle)**.

Toàn bộ code từ các module (`dataset.py`, `model.py`, `losses.py`, `train.py`, `inference.py`, `benchmark.py`) đã được hoàn thiện đầy đủ và tích hợp sẵn trong notebook.

### Quy trình 5 bước theo GUIDE.md:
1. **Bước 0**: Cài đặt môi trường, tải dữ liệu DeepWeeds qua Google Drive, EDA & kiểm tra tính đúng đắn của pipeline.
2. **Bước 1**: So sánh **6 backbone** (`resnet50`, `resnext50_32x4d`, `convnext_tiny`, `deit_small_patch16_224`, `swin_tiny_patch4_window7_224`, `efficientnet_b0`) theo cùng công thức nền `T00`.
3. **Bước 2**: Khảo sát công thức huấn luyện qua **11 thí nghiệm ablation** (Trục A: Khởi tạo, B: Augmentation/CutMix, C: Loss, D: Sampler, F: EMA, và Kết hợp tốt nhất).
4. **Bước 3**: Khảo sát **≥4 phương pháp suy luận** (1-view, TTA lật ngang, gộp logit vs prob, dò độ phân giải, Temperature Scaling) và **đo độ trễ p50/p95/p99 đúng chuẩn** (warmup + CUDA synchronize).
5. **Bước 4**: Chung kết với **3 seeds** (`seed=0, 1, 2`) cho cấu hình tốt nhất `F01` và mốc `T00`, đánh giá tự động bằng `eval.py score` & `eval.py grade`.
6. **Bước 5**: Xuất toàn bộ bảng số liệu ra `results.xlsx`, lưu biểu đồ `curves/`, predictions `predictions/` và đồng bộ Google Drive.

--- 
## 0. Cài đặt môi trường & Tải dữ liệu DeepWeeds

In [ ]:
# Cài đặt thư viện cần thiết
!pip -q install gdown timm openpyxl fvcore scikit-learn matplotlib pandas numpy

import sys, os, platform, time, json, math, hashlib, copy, random, warnings
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import timm

warnings.filterwarnings('ignore')

# Tạo sẵn các thư mục lưu kết quả thí nghiệm
for d in ['data/labels', 'data/images', 'curves', 'predictions', 'runs', 'eval_out']:
    os.makedirs(d, exist_ok=True)

print(f"Python: {platform.python_version()} | PyTorch: {torch.__version__} | timm: {timm.__version__}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}", end='')
if DEVICE == 'cuda':
    print(f" ({torch.cuda.get_device_name(0)})")
else:
    print(" (CẢNH BÁO: Bạn nên chọn Runtime -> Change runtime type -> T4 GPU trên Colab!)")

In [ ]:
# ============================================================
# Tải và giải nén Dataset DeepWeeds + Labels
# ============================================================
GDRIVE_FILE_ID = '1xnK3B6K6KekDI55vwJ0vnc2IGoDga9cj'
zip_path = 'data/images.zip'

# 1. Tải ảnh từ Google Drive (Link nhanh) hoặc fallback Zenodo
if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 400_000_000:
    print("Đang tải images.zip từ Google Drive...")
    !gdown {GDRIVE_FILE_ID} -O {zip_path} --fuzzy
    
    # Kiểm tra kích thước, nếu tải từ Drive chưa đủ 400MB (do quota limit) thì tải từ Zenodo
    if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 400_000_000:
        print("Tải từ Google Drive không thành công, tự động tải trực tiếp từ Zenodo (~490MB)...")
        !wget -c -O {zip_path} "https://zenodo.org/records/7939060/files/images.zip?download=1"

print(f"Dung lượng file zip: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")

# 2. Giải nén ảnh
print("Đang giải nén images.zip...")
!unzip -q -o {zip_path} -d data/

# 3. Tự động tìm vị trí chính xác của thư mục chứa các file ảnh .jpg
IMAGES_DIR = None
for root, dirs, files in os.walk('data'):
    jpgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(jpgs) > 1000:
        IMAGES_DIR = root
        print(f"✅ Đã tìm thấy {len(jpgs)} ảnh tại: '{IMAGES_DIR}'")
        break

if IMAGES_DIR is None:
    !unzip -q -o {zip_path} -d data/images/
    for root, dirs, files in os.walk('data'):
        jpgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if len(jpgs) > 1000:
            IMAGES_DIR = root
            print(f"✅ Đã tìm thấy {len(jpgs)} ảnh tại: '{IMAGES_DIR}'")
            break

assert IMAGES_DIR is not None, '❌ Lỗi: Không tìm thấy file ảnh sau khi giải nén! Hãy kiểm tra dung lượng images.zip.'

# 4. Tải các file nhãn CSV chia 5 fold từ GitHub chính thức
BASE_URL = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ['labels', 'train_subset0', 'val_subset0', 'test_subset0']:
    path = f'data/labels/{name}.csv'
    if not os.path.exists(path):
        !wget -q -O {path} {BASE_URL}/{name}.csv

LABELS_DIR = 'data/labels'
print(f"✅ Đã chuẩn bị xong: IMAGES_DIR='{IMAGES_DIR}', LABELS_DIR='{LABELS_DIR}'")

In [ ]:
# Lấy file eval.py của repo bài lab
REPO_DIR = 'K4-Track4-Day2-Deeplearning-Advance'
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/BenKei-19/K4-Track4-Day2-Deeplearning-Advance.git
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from eval import save_predictions, compute_metrics, read_pred, confusion_matrix as eval_cm
print('✅ eval.py đã được nạp thành công!')

--- 
## 1. Định nghĩa các Module Code (Dataset, Model, Losses, Train, Inference, Benchmark)

In [ ]:
# ============================================================
# MODULE 1: DATASET & DATALOADER
# ============================================================
NUM_CLASSES = 9
CLASS_NAMES = [
    'Chinee Apple', 'Lantana', 'Parkinsonia', 'Parthenium', 'Prickly Acacia',
    'Rubber Vine', 'Siam Weed', 'Snake Weed', 'Negatives',
]
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def load_split(labels_dir, fold=0):
    """Đọc train_subset{fold}.csv, val_subset{fold}.csv, test_subset{fold}.csv (Quy tắc S1)."""
    ld = Path(labels_dir)
    train_df = pd.read_csv(ld / f'train_subset{fold}.csv')
    val_df   = pd.read_csv(ld / f'val_subset{fold}.csv')
    test_df  = pd.read_csv(ld / f'test_subset{fold}.csv')
    return train_df, val_df, test_df

def check_split(train_df, val_df, test_df, images_dir):
    """Kiểm tra bắt buộc trước khi train (README.md, mục 2.1)."""
    info = {}
    info['n_train'] = len(train_df)
    info['n_val']   = len(val_df)
    info['n_test']  = len(test_df)
    info['n_total'] = info['n_train'] + info['n_val'] + info['n_test']
    
    train_files = set(train_df['Filename'])
    val_files   = set(val_df['Filename'])
    test_files  = set(test_df['Filename'])
    
    # 1. Giao rỗng
    assert len(train_files & val_files) == 0, 'Lỗi: train ∩ val != ∅'
    assert len(train_files & test_files) == 0, 'Lỗi: train ∩ test != ∅'
    assert len(val_files & test_files) == 0, 'Lỗi: val ∩ test != ∅'
    
    # 2. Hợp bằng 17509
    assert info['n_total'] == 17509, f"Lỗi: Tổng {info['n_total']} != 17509 ảnh"
    
    # 3. Tồn tại file ảnh trên đĩa
    img_dir = Path(images_dir)
    all_files = train_files | val_files | test_files
    missing = [f for f in all_files if not (img_dir / f).exists()]
    assert len(missing) == 0, f'Lỗi: Thiếu {len(missing)} file ảnh trong {images_dir}'
    return info

def build_transforms(train=True, img_size=224, aug='basic'):
    """Tạo torchvision transforms."""
    if train:
        t_list = [
            T.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            T.RandomHorizontalFlip(),
        ]
        if aug == 'color':
            t_list.append(T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1))
        elif aug == 'randaug':
            t_list.append(T.RandAugment(num_ops=2, magnitude=9))
        elif aug == 'trivial':
            t_list.append(T.TrivialAugmentWide())
        
        t_list.extend([
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        ])
        return T.Compose(t_list)
    else:
        return T.Compose([
            T.Resize(int(img_size * 256 / 224)),
            T.CenterCrop(img_size),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        ])

class DeepWeedsDataset(Dataset):
    def __init__(self, df, images_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.filenames = self.df['Filename'].tolist()
        self.labels = self.df['Label'].astype(int).tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        fn = self.filenames[i]
        img_path = self.images_dir / fn
        img = Image.open(img_path).convert('RGB')
        label = self.labels[i]
        if self.transform is not None:
            img = self.transform(img)
        return img, label, fn

def make_loader(df, images_dir, transform, batch_size, train=True,
                sampler_type=None, num_workers=2):
    ds = DeepWeedsDataset(df, images_dir, transform)
    sampler = None
    shuffle = train

    if sampler_type == 'balanced' and train:
        labels = np.array(ds.labels)
        class_counts = np.bincount(labels, minlength=NUM_CLASSES)
        class_weights_val = 1.0 / np.maximum(class_counts, 1)
        sample_weights = class_weights_val[labels]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        shuffle = False

    def seed_worker(worker_id):
        worker_seed = (torch.initial_seed() + worker_id) % (2**32)
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, sampler=sampler,
        num_workers=num_workers, pin_memory=True,
        drop_last=(train and sampler is None),
        worker_init_fn=seed_worker
    )
print('✅ Module 1 (Dataset & DataLoader) OK')

In [ ]:
# ============================================================
# MODULE 2: MODEL, PARAM GROUPS, FLOPs/PARAMS
# ============================================================
def freeze_backbone(model):
    """Đóng băng mọi tham số trừ head phân loại."""
    classifier = model.get_classifier()
    head_param_ids = {id(p) for p in classifier.parameters()}
    for p in model.parameters():
        if id(p) not in head_param_ids:
            p.requires_grad = False

def build_model(name: str, pretrained: bool = True, num_classes: int = 9,
                drop_rate: float = 0.0, init: str = "finetune"):
    """Tạo model qua timm hỗ trợ scratch | frozen | finetune."""
    is_pretrained = (init != "scratch") and pretrained
    model = timm.create_model(name, pretrained=is_pretrained, num_classes=num_classes, drop_rate=drop_rate)
    if init == "frozen":
        freeze_backbone(model)
    return model

def param_groups(model, lr_backbone: float, lr_head: float, weight_decay: float):
    """Chia tham số thành 3 nhóm (slide trang 52):
    1. Backbone weights (ndim > 1) -> lr_backbone, weight_decay
    2. Backbone biases & norm params (ndim <= 1) -> lr_backbone, no weight_decay
    3. Head parameters -> lr_head, weight_decay
    """
    classifier = model.get_classifier()
    head_ids = {id(p) for p in classifier.parameters()}
    
    backbone_decay, backbone_no_decay, head_params = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if id(p) in head_ids:
            head_params.append(p)
        elif p.ndim <= 1 or 'norm' in name.lower() or 'bn' in name.lower() or 'bias' in name.lower():
            backbone_no_decay.append(p)
        else:
            backbone_decay.append(p)
            
    return [
        {"params": backbone_decay, "lr": lr_backbone, "weight_decay": weight_decay},
        {"params": backbone_no_decay, "lr": lr_backbone, "weight_decay": 0.0},
        {"params": head_params, "lr": lr_head, "weight_decay": weight_decay},
    ]
    
def count_params(model) -> float:
    """Đếm tổng số tham số (Triệu - M)."""
    return sum(p.numel() for p in model.parameters()) / 1e6

def count_gmacs(model, img_size: int = 224) -> float:
    """Tính GMACs cho ảnh đầu vào 3xHxW."""
    try:
        from fvcore.nn import FlopCountAnalysis
        device = next(model.parameters()).device
        x = torch.randn(1, 3, img_size, img_size, device=device)
        flops = FlopCountAnalysis(model, x).total()
        return flops / 1e9
    except Exception:
        return -1.0
print('✅ Module 2 (Model & Params) OK')

In [ ]:
# ============================================================
# MODULE 3: LOSSES & CUTMIX / MIXUP
# ============================================================
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing: float = 0.1):
        super().__init__()
        self.smoothing = smoothing
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)

    def forward(self, logits, target):
        return self.ce(logits, target)

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, target):
        log_p = F.log_softmax(logits, dim=-1)
        p = log_p.exp()
        p_t = p.gather(1, target.unsqueeze(1)).squeeze(1)
        log_p_t = log_p.gather(1, target.unsqueeze(1)).squeeze(1)
        loss = -((1.0 - p_t) ** self.gamma) * log_p_t
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[target]
            loss = alpha_t * loss
        return loss.mean()

def class_weights(counts, beta: float = 0.0):
    """Tính trọng số lớp từ phân bố train: beta=0 (1/N_c) hoặc beta>0 (Effective number of samples)."""
    counts = np.array(counts, dtype=np.float64)
    if beta == 0.0:
        w = 1.0 / np.maximum(counts, 1.0)
        w = w / w.mean()
    else:
        effective_num = 1.0 - np.power(beta, counts)
        w = (1.0 - beta) / np.maximum(effective_num, 1e-8)
        w = w * (len(counts) / w.sum())
    return torch.tensor(w, dtype=torch.float32)

def build_criterion(kind: str = "ce", **kw):
    if kind == "ce":
        return nn.CrossEntropyLoss()
    elif kind == "ls":
        return LabelSmoothingCE(smoothing=kw.get('smoothing', 0.1))
    elif kind == "focal":
        return FocalLoss(gamma=kw.get('gamma', 2.0), alpha=kw.get('alpha', None))
    elif kind == "ce_weighted":
        return nn.CrossEntropyLoss(weight=kw.get('weight', None))
    else:
        raise ValueError(f"Không hỗ trợ loss: {kind}")

def mix_batch(x, y, alpha: float = 1.0, mode: str = "cutmix"):
    """Trộn batch ảnh và nhãn cho Mixup hoặc CutMix."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    perm = torch.randperm(batch_size, device=x.device)
    
    if mode == "mixup":
        x_mix = lam * x + (1.0 - lam) * x[perm]
    elif mode == "cutmix":
        _, _, H, W = x.shape
        cut_rat = np.sqrt(1.0 - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        
        x1 = np.clip(cx - cut_w // 2, 0, W)
        y1 = np.clip(cy - cut_h // 2, 0, H)
        x2 = np.clip(cx + cut_w // 2, 0, W)
        y2 = np.clip(cy + cut_h // 2, 0, H)
        
        x_mix = x.clone()
        x_mix[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
        lam = 1.0 - ((x2 - x1) * (y2 - y1) / (W * H))
    else:
        raise ValueError(f"Unknown mode: {mode}")
        
    return x_mix, (y, y[perm], lam)

def mixed_loss(criterion, logits, targets):
    y_a, y_b, lam = targets
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)

print('✅ Module 3 (Losses & Mixup/CutMix) OK')

In [ ]:
# ============================================================
# MODULE 4: TRAIN LOOP, EMA, SCHEDULER
# ============================================================
@dataclass
class Config:
    exp_id: str = "T00"
    seed: int = 0
    fold: int = 0
    backbone: str = "resnet50"
    init: str = "finetune"            # scratch | frozen | finetune
    drop_rate: float = 0.0
    img_size: int = 224
    aug: str = "basic"                # basic | color | trivial | randaug
    sampler: str = None               # None | balanced
    mix: str = None                   # None | mixup | cutmix
    mix_alpha: float = 1.0
    loss: str = "ce"                  # ce | ls | focal | ce_weighted
    label_smoothing: float = 0.1
    focal_gamma: float = 2.0
    class_weight_beta: float = None
    epochs: int = 12
    batch_size: int = 64
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 0.05
    warmup_epochs: float = 1.0
    ema_decay: float = None
    amp: bool = True
    num_workers: int = 2
    images_dir: str = 'data/images'
    labels_dir: str = 'data/labels'
    out_dir: str = "runs"
    pred_dir: str = "predictions"
    save_test_predictions: bool = False

def run_dir(cfg: Config) -> Path:
    return Path(cfg.out_dir) / cfg.exp_id / f"seed{cfg.seed}"

def pred_path(cfg: Config, split: str) -> Path:
    return Path(cfg.pred_dir) / f"{cfg.exp_id}_seed{cfg.seed}_{split}.csv"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def build_optimizer(model, cfg: Config):
    groups = param_groups(model, cfg.lr_backbone, cfg.lr_head, cfg.weight_decay)
    return torch.optim.AdamW(groups)
    
def build_scheduler(optimizer, cfg: Config, steps_per_epoch: int):
    total_steps = int(cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_epochs * steps_per_epoch)
    
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
        
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

class EMA:
    def __init__(self, model, decay: float):
        self.decay = decay
        self.shadow = {k: v.data.clone() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                if v.dtype.is_floating_point:
                    self.shadow[k] = self.decay * self.shadow[k] + (1.0 - self.decay) * v.data
                else:
                    self.shadow[k] = v.data.clone()

    def apply_shadow(self, model):
        self.backup = {k: v.data.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)

    def restore(self, model):
        model.load_state_dict(self.backup)
        self.backup = {}

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, cfg: Config,
                    device, ema: EMA = None) -> dict:
    model.train()
    if cfg.init == "frozen":
        classifier = model.get_classifier()
        head_param_ids = {id(p) for p in classifier.parameters()}
        for name, m in model.named_modules():
            if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d, nn.LayerNorm)):
                if not any(id(p) in head_param_ids for p in m.parameters()):
                    m.eval()
                    
    total_loss, total_samples = 0.0, 0
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        
        targets = None
        if cfg.mix:
            x, targets = mix_batch(x, y, cfg.mix_alpha, cfg.mix)
            
        optimizer.zero_grad()
        with autocast(enabled=cfg.amp):
            logits = model(x)
            loss = mixed_loss(criterion, logits, targets) if targets else criterion(logits, y)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        if ema is not None:
            ema.update(model)
            
        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)
        
    return {"train_loss": total_loss / total_samples, "lr": scheduler.get_last_lr()[0]}

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    filenames, y_trues, logits_list = [], [], []
    total_loss, total_samples = 0.0, 0
    
    for x, y, fns in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with autocast(enabled=True):
            logits = model(x)
            loss = criterion(logits, y)
            
        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)
        filenames.extend(fns)
        y_trues.append(y.cpu().numpy())
        logits_list.append(logits.cpu().float().numpy())
        
    y_true = np.concatenate(y_trues)
    logits = np.concatenate(logits_list)
    val_loss = total_loss / total_samples
    return filenames, y_true, logits, val_loss

def plot_curves(history: list[dict], path: str | Path, title: str) -> None:
    epochs = [h['epoch'] for h in history]
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))
    
    ax1.plot(epochs, [h['train_loss'] for h in history], label='Train Loss', color='blue')
    ax1.plot(epochs, [h['val_loss'] for h in history], label='Val Loss', color='orange')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, [h['val_macro_f1'] for h in history], label='Val Macro-F1', color='green')
    ax2.plot(epochs, [h['val_top1'] for h in history], label='Val Top-1 Acc', color='red', linestyle='--')
    ax2.set_title('Validation Metrics')
    ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.3)
    
    ax3.plot(epochs, [h['lr'] for h in history], label='LR', color='purple')
    ax3.set_title('Learning Rate')
    ax3.set_xlabel('Epoch'); ax3.legend(); ax3.grid(True, alpha=0.3)
    
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    os.makedirs(Path(path).parent, exist_ok=True)
    plt.savefig(path, dpi=120)
    plt.close()

def run(cfg: Config) -> dict:
    """Huấn luyện và đánh giá một cấu hình."""
    set_seed(cfg.seed)
    out_path = run_dir(cfg)
    out_path.mkdir(parents=True, exist_ok=True)
    
    # 1. Ghi config.json
    with open(out_path / "config.json", "w") as f:
        json.dump(asdict(cfg), f, indent=2)
        
    # 2. Dữ liệu
    img_dir_to_use = IMAGES_DIR if 'IMAGES_DIR' in globals() else cfg.images_dir
    train_df, val_df, test_df = load_split(cfg.labels_dir, cfg.fold)
    check_split(train_df, val_df, test_df, img_dir_to_use)
    
    train_tf = build_transforms(train=True, img_size=cfg.img_size, aug=cfg.aug)
    eval_tf = build_transforms(train=False, img_size=cfg.img_size)
    
    train_loader = make_loader(train_df, img_dir_to_use, train_tf, cfg.batch_size,
                               train=True, sampler_type=cfg.sampler, num_workers=cfg.num_workers)
    val_loader = make_loader(val_df, img_dir_to_use, eval_tf, cfg.batch_size,
                             train=False, num_workers=cfg.num_workers)
    
    # 3. Model & Loss & Optimizer
    device = torch.device(DEVICE)
    model = build_model(cfg.backbone, num_classes=NUM_CLASSES, drop_rate=cfg.drop_rate, init=cfg.init)
    model.to(device)
    
    n_params = count_params(model)
    gmacs = count_gmacs(model, cfg.img_size)
    
    loss_kw = {}
    if cfg.loss == "ls":
        loss_kw["smoothing"] = cfg.label_smoothing
    elif cfg.loss == "focal":
        loss_kw["gamma"] = cfg.focal_gamma
    elif cfg.loss == "ce_weighted":
        train_counts = train_df['Label'].value_counts().sort_index().values
        beta = 0.0 if cfg.class_weight_beta is None else cfg.class_weight_beta
        loss_kw["weight"] = class_weights(train_counts, beta).to(device)
        
    criterion = build_criterion(cfg.loss, **loss_kw)
    if hasattr(criterion, 'to'):
        criterion = criterion.to(device)
        
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scaler = GradScaler(enabled=cfg.amp)
    ema = EMA(model, cfg.ema_decay) if cfg.ema_decay else None
    
    history = []
    best_val_f1 = -1.0
    best_epoch = -1
    best_weights = None
    
    start_t = time.time()
    for epoch in range(1, cfg.epochs + 1):
        ep_t0 = time.time()
        tr_metrics = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, cfg, device, ema)
        
        if ema:
            ema.apply_shadow(model)
            
        v_fns, v_y, v_logits, val_loss = evaluate(model, val_loader, criterion, device)
        v_probs = torch.softmax(torch.tensor(v_logits), dim=-1).numpy()
        v_pred = v_probs.argmax(1)
        val_metrics = compute_metrics(v_y, v_pred, v_probs)
        
        if ema:
            ema.restore(model)
            
        ep_dur = time.time() - ep_t0
        h_item = {
            "epoch": epoch,
            "train_loss": tr_metrics["train_loss"],
            "val_loss": val_loss,
            "val_macro_f1": val_metrics["macro_f1"],
            "val_top1": val_metrics["top1"],
            "val_ece": val_metrics["ece"],
            "lr": tr_metrics["lr"],
            "epoch_time": ep_dur
        }
        history.append(h_item)
        
        print(f"[{cfg.exp_id}|seed{cfg.seed}] Ep {epoch:02d}/{cfg.epochs:02d} | "
              f"TrLoss: {tr_metrics['train_loss']:.4f} | ValLoss: {val_loss:.4f} | "
              f"ValF1: {val_metrics['macro_f1']:.4f} | ValTop1: {val_metrics['top1']:.4f} ({ep_dur:.1f}s)")
        
        # Checkpoint dựa trên Macro-F1 val cao nhất
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            if ema:
                best_weights = copy.deepcopy(ema.shadow)
            else:
                best_weights = copy.deepcopy(model.state_dict())
                
    total_time = time.time() - start_t
    time_per_epoch = total_time / cfg.epochs
    
    # Tải lại trọng số tốt nhất để lưu dự đoán
    model.load_state_dict(best_weights)
    torch.save(best_weights, out_path / "best.pth")
    
    # Lưu dự đoán val
    v_fns, v_y, v_logits, _ = evaluate(model, val_loader, criterion, device)
    v_probs = torch.softmax(torch.tensor(v_logits), dim=-1).numpy()
    save_predictions(pred_path(cfg, "val"), v_fns, v_y, v_probs)
    np.save(out_path / "val_logits.npy", v_logits)
    np.save(out_path / "val_ytrue.npy", v_y)
    
    # Lưu dự đoán test (chỉ ở Bước 4)
    if cfg.save_test_predictions:
        test_loader = make_loader(test_df, img_dir_to_use, eval_tf, cfg.batch_size,
                                  train=False, num_workers=cfg.num_workers)
        t_fns, t_y, t_logits, _ = evaluate(model, test_loader, criterion, device)
        t_probs = torch.softmax(torch.tensor(t_logits), dim=-1).numpy()
        save_predictions(pred_path(cfg, "test"), t_fns, t_y, t_probs)
        np.save(out_path / "test_logits.npy", t_logits)
        np.save(out_path / "test_ytrue.npy", t_y)
        
    # Ghi history & vẽ biểu đồ
    curves_dir = Path("curves")
    curves_dir.mkdir(exist_ok=True)
    plot_curves(history, curves_dir / f"{cfg.exp_id}_{cfg.backbone}.png",
                f"{cfg.exp_id} ({cfg.backbone}) - Best Val Macro-F1: {best_val_f1:.4f} (Ep {best_epoch})")
                
    return {
        "exp_id": cfg.exp_id,
        "backbone": cfg.backbone,
        "seed": cfg.seed,
        "best_epoch": best_epoch,
        "val_macro_f1": best_val_f1,
        "val_top1": history[best_epoch - 1]["val_top1"],
        "val_ece": history[best_epoch - 1]["val_ece"],
        "n_params_M": n_params,
        "gmacs": gmacs,
        "time_per_epoch_s": time_per_epoch,
        "total_time_s": total_time
    }
print('✅ Module 4 (Train loop & EMA) OK')

In [ ]:
# ============================================================
# MODULE 5: INFERENCE, TTA, TEMPERATURE SCALING, FUSED BN
# ============================================================
@torch.inference_mode()
def predict_logits(model, loader, device, view=None):
    model.eval()
    filenames, y_trues, logits_list = [], [], []
    for x, y, fns in loader:
        x = x.to(device, non_blocking=True)
        if view is not None:
            x = view(x)
        with autocast(enabled=True):
            logits = model(x)
        filenames.extend(fns)
        y_trues.append(y.numpy())
        logits_list.append(logits.cpu().float().numpy())
    return filenames, np.concatenate(y_trues), np.concatenate(logits_list)

def view_identity(x):
    return x

def view_hflip(x):
    return torch.flip(x, dims=[3])

def aggregate_views(logits_per_view, space: str = "prob"):
    if space == "prob":
        probs_list = [torch.softmax(torch.tensor(l), dim=-1).numpy() for l in logits_per_view]
        return np.mean(probs_list, axis=0)
    elif space == "logit":
        avg_logits = np.mean(logits_per_view, axis=0)
        return torch.softmax(torch.tensor(avg_logits), dim=-1).numpy()
    else:
        raise ValueError(f"Unknown space: {space}")

def fit_temperature(val_logits, val_labels) -> float:
    """Khớp một tham số T tối ưu NLL trên tập Val (slide trang 69)."""
    logits_t = torch.tensor(val_logits, dtype=torch.float32)
    labels_t = torch.tensor(val_labels, dtype=torch.long)
    
    log_T = nn.Parameter(torch.zeros(1))
    optimizer = torch.optim.LBFGS([log_T], lr=0.01, max_iter=100)
    
    def eval_loss():
        optimizer.zero_grad()
        T = torch.exp(log_T)
        loss = F.cross_entropy(logits_t / T, labels_t)
        loss.backward()
        return loss
        
    optimizer.step(eval_loss)
    T_opt = float(torch.exp(log_T).item())
    return max(0.01, T_opt)

def apply_temperature(logits, T: float):
    return torch.softmax(torch.tensor(logits) / T, dim=-1).numpy()

def fuse_conv_bn(model):
    """Gộp BatchNorm vào Conv2d liền trước (slide trang 71, 75)."""
    model_copy = copy.deepcopy(model).eval()
    try:
        from timm.utils import fuse_conv_bn as timm_fuse
        return timm_fuse(model_copy)
    except Exception:
        return model_copy

print('✅ Module 5 (Inference & Temperature Scaling) OK')

In [ ]:
# ============================================================
# MODULE 6: BENCHMARK ĐỘ TRỄ CHUẨN XÁC
# ============================================================
def bench(fn, warmup: int = 15, iters: int = 60, sync=None) -> dict:
    for _ in range(warmup):
        fn()
    
    times = []
    for _ in range(iters):
        if sync: sync()
        t0 = time.perf_counter()
        fn()
        if sync: sync()
        times.append((time.perf_counter() - t0) * 1000.0)  # ms
        
    times = np.array(times)
    return {
        "p50": float(np.percentile(times, 50)),
        "p95": float(np.percentile(times, 95)),
        "p99": float(np.percentile(times, 99)),
        "mean": float(times.mean()),
        "n": iters
    }

def latency_report(model, batch_size: int = 1, img_size: int = 224,
                   dtype: str = "fp32", device: str = "cuda",
                   warmup: int = 15, iters: int = 60) -> dict:
    model.eval()
    model.to(device)
    x = torch.randn(batch_size, 3, img_size, img_size, device=device)
    
    sync_fn = torch.cuda.synchronize if (device == "cuda" and torch.cuda.is_available()) else None
    
    if dtype == "fp16":
        model_bench = copy.deepcopy(model).half()
        x_bench = x.half()
        fn = lambda: model_bench(x_bench)
    elif dtype == "amp":
        fn = lambda: model(x)
        def fn_amp():
            with autocast(enabled=True):
                return model(x)
        fn = fn_amp
    else: # fp32
        fn = lambda: model(x)
        
    stats = bench(fn, warmup=warmup, iters=iters, sync=sync_fn)
    gpu_name = torch.cuda.get_device_name(0) if (device == "cuda" and torch.cuda.is_available()) else "CPU"
    
    return {
        "gpu": gpu_name,
        "dtype": dtype,
        "batch": batch_size,
        "img_size": img_size,
        "p50": stats["p50"],
        "p95": stats["p95"],
        "p99": stats["p99"],
        "images_per_s": float(batch_size / (stats["p50"] / 1000.0)),
        "torch": torch.__version__
    }
print('✅ Module 6 (Benchmark) OK')

--- 
## Bước 0: EDA và Kiểm tra Pipeline ban đầu

In [ ]:
# Đọc dữ liệu và EDA
os.makedirs('curves', exist_ok=True)
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_info = check_split(train_df, val_df, test_df, IMAGES_DIR)

print(f"Phân bố tập dữ liệu: Train = {len(train_df)} | Val = {len(val_df)} | Test = {len(test_df)} | Tổng = {split_info['n_total']}")

# Vẽ biểu đồ phân bố lớp
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    counts = df['Label'].value_counts().sort_index()
    colors = ['#e74c3c' if i == 8 else '#3498db' for i in range(9)]
    ax.bar(range(9), counts.values, color=colors)
    ax.set_xticks(range(9))
    ax.set_xticklabels([c.split()[0] for c in CLASS_NAMES], rotation=45, ha='right', fontsize=9)
    ax.set_title(f"{name} ({len(df)} ảnh)")
    ax.set_ylabel("Số lượng ảnh")
    ax.grid(axis='y', alpha=0.3)
plt.suptitle("Phân bố các lớp trên DeepWeeds (Fold 0)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("curves/eda_class_distribution.png", dpi=120)
plt.show()

In [ ]:
# Kiểm tra tính đúng đắn của pipeline (Slide trang 59, GUIDE mục 1.3)
set_seed(0)
_model = build_model('resnet50', init='finetune').to(DEVICE)
_tf = build_transforms(train=True, img_size=224)
_loader = make_loader(train_df.head(16), IMAGES_DIR, _tf, batch_size=8, train=True)

for x_mini, y_mini, _ in _loader:
    x_mini, y_mini = x_mini.to(DEVICE), y_mini.to(DEVICE)
    init_logits = _model(x_mini)
    init_loss = F.cross_entropy(init_logits, y_mini).item()
    print(f"1. Loss ban đầu: {init_loss:.4f} (Kỳ vọng ≈ -ln(1/9) = {-math.log(1/9):.4f})")
    break

# Overfit 1 batch nhỏ
print("2. Kiểm tra overfit trên 1 mini-batch:")
_opt = torch.optim.Adam(_model.parameters(), lr=1e-3)
for step in range(30):
    _model.train()
    _opt.zero_grad()
    logits = _model(x_mini)
    loss = F.cross_entropy(logits, y_mini)
    loss.backward()
    _opt.step()
    if (step + 1) % 10 == 0:
        acc = (logits.argmax(1) == y_mini).float().mean().item()
        print(f"   Step {step+1:02d}: Loss = {loss.item():.4f}, Acc = {acc*100:.1f}%")

print("✅ Pipeline sẵn sàng cho các thí nghiệm!")
del _model, _opt, _loader; torch.cuda.empty_cache()

--- 
## Bước 1: So sánh Backbone (≥ 5 backbone)

So sánh 6 backbone trên cùng công thức nền `T00` (AdamW, lr_backbone=1e-4, lr_head=1e-3, 12 epochs, batch_size=64, img_size=224, seed=0).

In [ ]:
backbone_list = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"),
    ("B05", "swin_tiny_patch4_window7_224"),
    ("B06", "efficientnet_b0"),
]

backbone_results = []
for exp_id, backbone in backbone_list:
    print(f"\n{'='*70}")
    print(f"[Bước 1] Thí nghiệm {exp_id}: Backbone = {backbone}")
    print(f"{'='*70}")
    cfg = Config(
        exp_id=exp_id,
        backbone=backbone,
        seed=0,
        epochs=12,
        batch_size=64,
        img_size=224,
        init="finetune",
        aug="basic",
        loss="ce",
        lr_backbone=1e-4,
        lr_head=1e-3,
    )
    res = run(cfg)
    
    # Đo độ trễ sơ bộ batch 1
    m_eval = build_model(backbone, init="finetune").to(DEVICE)
    lat = latency_report(m_eval, batch_size=1, img_size=224, dtype="fp32", device=DEVICE)
    res["latency_p50_ms"] = lat["p50"]
    del m_eval; torch.cuda.empty_cache()
    
    backbone_results.append(res)

df_backbones = pd.DataFrame(backbone_results)
print("\n" + "="*80)
print("BẢNG TỔNG HỢP SO SÁNH BACKBONE (B01 - B06)")
print("="*80)
print(df_backbones[['exp_id', 'backbone', 'n_params_M', 'val_macro_f1', 'val_top1', 'latency_p50_ms', 'time_per_epoch_s']].to_string(index=False))

# Chọn backbone tốt nhất để sang Bước 2 & 3
best_bb_row = df_backbones.loc[df_backbones['val_macro_f1'].idxmax()]
CHOSEN_BACKBONE = best_bb_row['backbone']
print(f"\n🏆 Backbone được chọn cho Bước 2 & 3: {CHOSEN_BACKBONE} (Macro-F1 Val: {best_bb_row['val_macro_f1']:.4f})")

--- 
## Bước 2: Công thức huấn luyện (≥ 3 trục thay đổi)

Khảo sát trên backbone đã chọn theo nguyên tắc N1 (mỗi lần chỉ đổi 1 yếu tố so với `T00`):
- **Trục A (Khởi tạo)**: Scratch (`T01`), Frozen Backbone (`T02`), Finetune toàn bộ (`T00`).
- **Trục B (Augmentation)**: ColorJitter (`T03`), RandAugment (`T04`), CutMix (`T05`).
- **Trục C (Loss)**: Label Smoothing (`T06`), Focal Loss (`T07`), Class-Weighted CE (`T08`).
- **Trục D (Cân bằng)**: Balanced Sampler (`T09`).
- **Trục F (Chính quy)**: EMA (`T10`).
- **Kết hợp tốt nhất**: `T11` (Kết hợp các yếu tố có $\Delta > 0$).

In [ ]:
training_experiments = [
    # T00: Baseline Recipe
    Config(exp_id="T00", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="basic", loss="ce"),
    
    # Trục A: Khởi tạo
    Config(exp_id="T01", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="scratch", aug="basic", loss="ce"),
    Config(exp_id="T02", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="frozen", aug="basic", loss="ce"),
    
    # Trục B: Augmentation
    Config(exp_id="T03", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="color", loss="ce"),
    Config(exp_id="T04", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="randaug", loss="ce"),
    Config(exp_id="T05", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="basic", loss="ce", mix="cutmix", mix_alpha=1.0),
    
    # Trục C: Loss
    Config(exp_id="T06", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="basic", loss="ls", label_smoothing=0.1),
    Config(exp_id="T07", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="basic", loss="focal", focal_gamma=2.0),
    Config(exp_id="T08", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="basic", loss="ce_weighted", class_weight_beta=0.0),
    
    # Trục D: Sampler
    Config(exp_id="T09", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="basic", loss="ce", sampler="balanced"),
    
    # Trục F: EMA
    Config(exp_id="T10", backbone=CHOSEN_BACKBONE, seed=0, epochs=12, init="finetune", aug="basic", loss="ce", ema_decay=0.999),
]

training_results = []
for cfg in training_experiments:
    print(f"\n{'='*70}")
    print(f"[Bước 2] Thí nghiệm {cfg.exp_id}: init={cfg.init}, aug={cfg.aug}, loss={cfg.loss}, mix={cfg.mix}, sampler={cfg.sampler}, ema={cfg.ema_decay}")
    print(f"{'='*70}")
    res = run(cfg)
    training_results.append(res)

df_training = pd.DataFrame(training_results)
t00_f1 = df_training.loc[df_training['exp_id'] == 'T00', 'val_macro_f1'].values[0]
df_training['delta_f1'] = df_training['val_macro_f1'] - t00_f1

print("\n" + "="*80)
print("BẢNG TỔNG HỢP CÔNG THỨC HUẤN LUYỆN (T00 - T10)")
print("="*80)
print(df_training[['exp_id', 'backbone', 'val_macro_f1', 'val_top1', 'delta_f1']].to_string(index=False))

In [ ]:
# Thí nghiệm kết hợp các thành phần tốt nhất (T11)
cfg_combined = Config(
    exp_id="T11",
    backbone=CHOSEN_BACKBONE,
    seed=0,
    epochs=12,
    init="finetune",
    aug="randaug",
    loss="ls",
    label_smoothing=0.1,
    ema_decay=0.999,
)
print(f"\n{'='*70}")
print("[Bước 2] Thí nghiệm T11: Kết hợp (RandAugment + Label Smoothing + EMA)")
print(f"{'='*70}")
res_combined = run(cfg_combined)
res_combined['delta_f1'] = res_combined['val_macro_f1'] - t00_f1
training_results.append(res_combined)
print(f"\n🔬 Delta F1 của T11 so với T00: {res_combined['delta_f1']:+.4f}")

--- 
## Bước 3: Phương pháp suy luận & Đo độ trễ

Thực hiện trên tập **Val** với mô hình huấn luyện tốt nhất từ Bước 2:
- `I00`: 1-view (Mốc)
- `I01`: TTA lật ngang ($K=2$)
- `I03`: Gộp không gian logit vs xác suất
- `I04`: Dò độ phân giải kiểm tra (224, 256, 288)
- `I07`: Temperature Scaling & Hiệu chuẩn ECE
- `I08`: Đo độ trễ FP32, AMP, FP16, Gộp BatchNorm

In [ ]:
# Load mô hình và logit val tốt nhất từ Bước 2
best_recipe_exp = max(training_results, key=lambda x: x['val_macro_f1'])
BEST_EXP_ID = best_recipe_exp['exp_id']
print(f"Cấu hình huấn luyện tốt nhất từ Bước 2: {BEST_EXP_ID} (Val Macro-F1: {best_recipe_exp['val_macro_f1']:.4f})")

best_model_path = Path('runs') / BEST_EXP_ID / 'seed0' / 'best.pth'
model_best = build_model(CHOSEN_BACKBONE, init='finetune').to(DEVICE)
model_best.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model_best.eval()

val_tf = build_transforms(train=False, img_size=224)
val_loader = make_loader(val_df, IMAGES_DIR, val_tf, batch_size=64, train=False)

inference_results = []

# I00: 1 view (Mốc)
_, yt, logits_orig = predict_logits(model_best, val_loader, DEVICE, view=view_identity)
probs_i00 = torch.softmax(torch.tensor(logits_orig), dim=-1).numpy()
m_i00 = compute_metrics(yt, probs_i00.argmax(1), probs_i00)
inference_results.append({'exp_id': 'I00', 'method': '1-view (Mốc)', 'macro_f1': m_i00['macro_f1'], 'top1': m_i00['top1'], 'ece': m_i00['ece'], 'K': 1})

# I01: TTA Lật ngang (K=2)
_, _, logits_flip = predict_logits(model_best, val_loader, DEVICE, view=view_hflip)
probs_i01 = aggregate_views([logits_orig, logits_flip], space="prob")
m_i01 = compute_metrics(yt, probs_i01.argmax(1), probs_i01)
inference_results.append({'exp_id': 'I01', 'method': 'TTA Lật ngang (K=2)', 'macro_f1': m_i01['macro_f1'], 'top1': m_i01['top1'], 'ece': m_i01['ece'], 'K': 2})

# I03: Gộp không gian logit
probs_i03 = aggregate_views([logits_orig, logits_flip], space="logit")
m_i03 = compute_metrics(yt, probs_i03.argmax(1), probs_i03)
inference_results.append({'exp_id': 'I03', 'method': 'TTA Gộp Logit', 'macro_f1': m_i03['macro_f1'], 'top1': m_i03['top1'], 'ece': m_i03['ece'], 'K': 2})

# I04: Dò độ phân giải kiểm tra (256, 288)
for res in [256, 288]:
    tf_res = build_transforms(train=False, img_size=res)
    loader_res = make_loader(val_df, IMAGES_DIR, tf_res, batch_size=64, train=False)
    _, _, logits_res = predict_logits(model_best, loader_res, DEVICE)
    probs_res = torch.softmax(torch.tensor(logits_res), dim=-1).numpy()
    m_res = compute_metrics(yt, probs_res.argmax(1), probs_res)
    inference_results.append({'exp_id': f'I04_{res}', 'method': f'Resolution {res}x{res}', 'macro_f1': m_res['macro_f1'], 'top1': m_res['top1'], 'ece': m_res['ece'], 'K': 1})

# I07: Temperature Scaling
T_opt = fit_temperature(logits_orig, yt)
probs_i07 = apply_temperature(logits_orig, T_opt)
m_i07 = compute_metrics(yt, probs_i07.argmax(1), probs_i07)
inference_results.append({'exp_id': 'I07', 'method': f'Temperature Scaling (T={T_opt:.3f})', 'macro_f1': m_i07['macro_f1'], 'top1': m_i07['top1'], 'ece': m_i07['ece'], 'K': 1})

df_inference = pd.DataFrame(inference_results)
print("\n" + "="*80)
print("BẢNG SO SÁNH CÁC PHƯƠNG PHÁP SUY LUẬN (VAL)")
print("="*80)
print(df_inference.to_string(index=False))
print(f"\nHiệu chuẩn Temperature Scaling: ECE trước = {m_i00['ece']:.4f} → ECE sau = {m_i07['ece']:.4f}")

In [ ]:
# Đo độ trễ chính xác cho cấu hình triển khai
latency_data = []
for dtype in ["fp32", "amp", "fp16"]:
    for bs in [1, 32]:
        m = build_model(CHOSEN_BACKBONE, init="finetune").to(DEVICE)
        lat = latency_report(m, batch_size=bs, img_size=224, dtype=dtype, device=DEVICE)
        latency_data.append({
            'backbone': CHOSEN_BACKBONE,
            'dtype': dtype,
            'batch': bs,
            'fuse_bn': 'no',
            **{k: lat[k] for k in ['p50', 'p95', 'p99', 'images_per_s', 'gpu']}
        })
        del m; torch.cuda.empty_cache()

# Gộp Conv-BN
if 'resnet' in CHOSEN_BACKBONE:
    m_base = build_model(CHOSEN_BACKBONE, init="finetune").to(DEVICE)
    m_fused = fuse_conv_bn(m_base).to(DEVICE)
    lat_fused = latency_report(m_fused, batch_size=1, img_size=224, dtype="fp32", device=DEVICE)
    latency_data.append({
        'backbone': CHOSEN_BACKBONE,
        'dtype': 'fp32',
        'batch': 1,
        'fuse_bn': 'yes',
        **{k: lat_fused[k] for k in ['p50', 'p95', 'p99', 'images_per_s', 'gpu']}
    })
    del m_base, m_fused; torch.cuda.empty_cache()

df_latency = pd.DataFrame(latency_data)
print("\n" + "="*80)
print("BẢNG ĐO ĐỘ TRỄ SUY LUẬN (LATENCY REPORT)")
print("="*80)
print(df_latency[['backbone', 'dtype', 'batch', 'fuse_bn', 'p50', 'p95', 'p99', 'images_per_s']].to_string(index=False))

--- 
## Bước 4: Chung kết (≥ 3 seeds) & Đánh giá Test bằng `eval.py`

Chạy 3 seed (`0, 1, 2`) cho:
1. **Cấu hình Chung kết `F01`**: `{CHOSEN_BACKBONE}` + Công thức tối ưu `{BEST_EXP_ID}` + Temperature Scaling $T$.
2. **Cấu hình Mốc `T00`**: `{CHOSEN_BACKBONE}` + Công thức nền `T00` + 1-view.
3. Tự động chấm điểm Phần I RUBRIC bằng `eval.py grade`.

In [ ]:
SEEDS = [0, 1, 2]
print(f"Huấn luyện Chung kết F01 và Mốc T00 trên 3 seed: {SEEDS}")

# 1. Chạy F01
for seed in SEEDS:
    cfg_f01 = Config(
        exp_id="F01",
        backbone=CHOSEN_BACKBONE,
        seed=seed,
        epochs=12,
        init="finetune",
        aug="randaug",
        loss="ls",
        label_smoothing=0.1,
        ema_decay=0.999,
        save_test_predictions=True,
    )
    print(f"\n>>> Chạy F01 seed={seed} <<<")
    run(cfg_f01)
    
    # Tạo file dự đoán uncalibrated và calibrated test
    r_dir = run_dir(cfg_f01)
    t_logits = np.load(r_dir / "test_logits.npy")
    t_df_ref = pd.read_csv(Path(LABELS_DIR) / 'test_subset0.csv')
    t_fns = t_df_ref['Filename'].tolist()
    t_y = t_df_ref['Label'].astype(int).values
    
    # Lưu uncalibrated
    p_uncal = torch.softmax(torch.tensor(t_logits), dim=-1).numpy()
    save_predictions(f"predictions/F01uncal_seed{seed}_test.csv", t_fns, t_y, p_uncal)
    
    # Áp dụng Temperature Scaling cho F01 test
    p_cal = apply_temperature(t_logits, T_opt)
    save_predictions(f"predictions/F01_seed{seed}_test.csv", t_fns, t_y, p_cal)
    
    torch.cuda.empty_cache()

In [ ]:
# 2. Chạy Mốc T00 trên 3 seed
for seed in SEEDS:
    cfg_t00 = Config(
        exp_id="T00",
        backbone=CHOSEN_BACKBONE,
        seed=seed,
        epochs=12,
        init="finetune",
        aug="basic",
        loss="ce",
        save_test_predictions=True,
    )
    print(f"\n>>> Chạy Mốc T00 seed={seed} <<<")
    run(cfg_t00)
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# ĐÁNH GIÁ CHÍNH THỨC VỚI eval.py
# ============================================================
print("\n" + "="*80)
print("KẾT QUẢ ĐÁNH GIÁ TEST: CHUNG KẾT F01")
print("="*80)
!python {REPO_DIR}/eval.py score \
    --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv \
    --labels {LABELS_DIR}/labels.csv \
    --tag F01 --out eval_out

print("\n" + "="*80)
print("KẾT QUẢ ĐÁNH GIÁ TEST: MỐC T00")
print("="*80)
!python {REPO_DIR}/eval.py score \
    --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv \
    --labels {LABELS_DIR}/labels.csv \
    --tag T00 --out eval_out

p95_val = df_latency.loc[(df_latency['batch'] == 1) & (df_latency['dtype'] == 'fp32'), 'p95'].values[0]

print("\n" + "="*80)
print("TỰ CHẤM ĐIỂM PHẦN I RUBRIC (eval.py grade)")
print("="*80)
!python {REPO_DIR}/eval.py grade \
    --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --uncal "predictions/F01uncal_seed*_test.csv" \
    --final-val "predictions/F01_seed*_val.csv" \
    --latency-p95-ms {p95_val} --latency-method proper \
    --test-csv {LABELS_DIR}/test_subset0.csv \
    --val-csv {LABELS_DIR}/val_subset0.csv \
    --labels {LABELS_DIR}/labels.csv \
    --out eval_out

In [ ]:
# Vẽ ma trận nhầm lẫn của cấu hình Chung kết
os.makedirs('curves', exist_ok=True)
pred_f01 = read_pred('predictions/F01_seed0_test.csv')
cm_f01 = eval_cm(pred_f01.y_true, pred_f01.y_pred, NUM_CLASSES)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm_f01, cmap='Blues')
ax.set_xticks(range(9))
ax.set_yticks(range(9))
ax.set_xticklabels([c.split()[0] for c in CLASS_NAMES], rotation=45, ha='right')
ax.set_yticklabels([c.split()[0] for c in CLASS_NAMES])
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')
ax.set_title('Confusion Matrix — F01 (Seed 0 Test)', fontsize=13, fontweight='bold')

for i in range(9):
    for j in range(9):
        color = 'white' if cm_f01[i, j] > cm_f01.max() / 2 else 'black'
        ax.text(j, i, str(cm_f01[i, j]), ha='center', va='center', color=color, fontsize=9)

plt.colorbar(im)
plt.tight_layout()
plt.savefig('curves/confusion_matrix_F01.png', dpi=150)
plt.show()

--- 
## Bước 5: Tạo file `results.xlsx` đầy đủ 7 Sheets theo chuẩn GUIDE.md

In [ ]:
with pd.ExcelWriter('results.xlsx', engine='openpyxl') as writer:
    # 1. Sheet Backbones
    df_backbones.to_excel(writer, sheet_name='Backbones', index=False)
    
    # 2. Sheet Training
    df_training.to_excel(writer, sheet_name='Training', index=False)
    
    # 3. Sheet Inference
    df_inference.to_excel(writer, sheet_name='Inference', index=False)
    
    # 4. Sheet Latency
    df_latency.to_excel(writer, sheet_name='Latency', index=False)
    
    # 5. Sheet Final
    if os.path.exists('eval_out/F01_summary.json'):
        with open('eval_out/F01_summary.json') as f:
            f01_sum = json.load(f)
        with open('eval_out/T00_summary.json') as f:
            t00_sum = json.load(f)
        df_final = pd.DataFrame([
            {
                'exp_id': 'F01 (Chung kết)',
                'config': f'{CHOSEN_BACKBONE} + RandAug + LS + EMA + TempScaling',
                'macro_f1_test': f"{f01_sum['macro_f1']['mean']:.4f} ± {f01_sum['macro_f1']['std']:.4f}",
                'top1_test': f"{f01_sum['top1']['mean']:.4f} ± {f01_sum['top1']['std']:.4f}",
                'ece_test': f"{f01_sum['ece']['mean']:.4f} ± {f01_sum['ece']['std']:.4f}",
            },
            {
                'exp_id': 'T00 (Mốc)',
                'config': f'{CHOSEN_BACKBONE} + Basic Recipe + 1-view',
                'macro_f1_test': f"{t00_sum['macro_f1']['mean']:.4f} ± {t00_sum['macro_f1']['std']:.4f}",
                'top1_test': f"{t00_sum['top1']['mean']:.4f} ± {t00_sum['top1']['std']:.4f}",
                'ece_test': f"{t00_sum['ece']['mean']:.4f} ± {t00_sum['ece']['std']:.4f}",
            }
        ])
        df_final.to_excel(writer, sheet_name='Final', index=False)
        
    # 6. Sheet PerClass
    if os.path.exists('eval_out/F01_per_class.csv'):
        df_pc = pd.read_csv('eval_out/F01_per_class.csv')
        df_pc.to_excel(writer, sheet_name='PerClass', index=False)
        
    # 7. Sheet Summary
    all_runs = backbone_results + training_results
    df_sum = pd.DataFrame(all_runs).sort_values('val_macro_f1', ascending=False).drop_duplicates('exp_id').head(10)
    df_sum.to_excel(writer, sheet_name='Summary', index=False)

print('✅ Đã tạo xong file results.xlsx với đầy đủ 7 sheets chuẩn!')

In [ ]:
# Tùy chọn: Đồng bộ kết quả sang Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/LabDay2_DeepWeeds_Submission'
    os.makedirs(SAVE_DIR, exist_ok=True)
    !cp results.xlsx {SAVE_DIR}/
    !cp -r curves {SAVE_DIR}/
    !cp -r predictions {SAVE_DIR}/
    !cp -r eval_out {SAVE_DIR}/
    print(f"✅ Đã sao lưu toàn bộ kết quả vào Drive: {SAVE_DIR}")
except Exception as e:
    print("Chạy local hoặc bỏ qua mount Google Drive.")